# Assignment 2
Trey Patrick

In [ ]:
## Question 1: You will scrape data from the above website in order to
## measure the accuracy of experts. To do so, you must first finish the following
## function in Python, which will be reused many times in the following question:

def collect_data(url):
    import requests
    import re
    import pandas as pd
    r = requests.get(url)
    html_code = r.text

    hits = re.finditer(r"<td class=\"game\">.+?</td>\s*<td class=\"pick\">.+?</td>", html_code)

    teams = []
    picks = []
    winners = []
    for hit in hits:
        row = hit.group()

        # Team name sits between the link and </a>; the winner's name is wrapped in <b></b>
        team = re.search(r"html\">(.+?)</a>", row).group(1)
        if re.search(r"<b>", team) is not None:
            winners.append(True)
        else:
            winners.append(False)
        team = re.sub(r"<b>", "", team)
        team = re.sub(r"</b>", "", team)
        teams.append(team)

        # Probability: the digits right before the % sign
        pick = re.search(r"(\d+)%", row).group(1)
        picks.append(int(pick))

    # Home team is always listed first, so the rows alternate: home, away, home, away, ...
    rows = []
    for index in range(0, len(teams), 2):
        home_team = teams[index]
        away_team = teams[index + 1]
        if winners[index]:
            winner_team = home_team
        else:
            winner_team = away_team
        rows.append([home_team, away_team, winner_team, picks[index], picks[index + 1]])

    result = pd.DataFrame(rows, columns = ["Home_Team", "Away_Team", "Winner_Team",
                                           "Prediction_Home_Team", "Prediction_Away_Team"])
    return result

In [ ]:
collect_data("https://rayna-xu-is.github.io/probabilityfootball/page1.html")

In [ ]:
## Question 2: Now that you are able to scrape relevant data from a single
## week, it is time to use the created function to collect and aggregate data for many
## weeks. In particular, you will collect experts’ predictions for all the 256 games (17
## weeks) in the 2008 NFL regular season. To do so, apply the collect_data function to
## every single webpage such as:

import pandas as pd

weekly_data = []
for week in range(1, 18):
    weekly_data.append(collect_data(f"https://rayna-xu-is.github.io/probabilityfootball/page{week}.html"))

season_data = pd.concat(weekly_data, ignore_index = True)
season_data.shape



In [ ]:
## Question 3

from scipy import stats

predictions_winner = []
for index in range(len(season_data)):
    if season_data["Winner_Team"][index] == season_data["Home_Team"][index]:
        predictions_winner.append(season_data["Prediction_Home_Team"][index])
    else:
        predictions_winner.append(season_data["Prediction_Away_Team"][index])

print("Number of games:", len(predictions_winner))
print("Mean prediction for the winning team:", sum(predictions_winner) / len(predictions_winner))
stats.ttest_1samp(predictions_winner, 50, alternative = "greater")

Which forecaster is more accurate? 
The experts are. On average they gave the winning team about a 57% chance, while a coin would only give 50%. The p value is less than 0.05 so the data is significant in identifying that the experts are better.  . 

Should the group use expert-driven forecasts instead of model-based forecasts?
  No because all this shows is that the experts are better than guessing randomly.
   It doesn't show they're better than an actual model built on past data. 

Is an unbiased coin an appropriate baseline? 
A coin flip is too weak. There are many things that go into match ups in the nfl. A coin flip might make you right 50% of the time but with stats and research you could be right more often. 
